# Revisión capa GOLD (marts Power BI + feature stores ML)

Los 6 marts para dashboards usan **grano agregado** (resúmenes por
fecha/hora/zona/servicio); el detalle viaje a viaje vive en silver/star/facts.
**Prueba de no-pérdida**: `SUM(viajes)` de cada mart == filas de los facts, exacto.

In [ ]:
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq

DATA = Path("../data") if Path("../data").exists() else Path("data")
CATS = ["green", "yellow", "fhv", "fhvhv"]
YEARS = [2023, 2024, 2025]
MESES = [(y, m) for y in YEARS for m in range(1, 13)]
pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(25)


def parquet_rows(path: Path) -> int | None:
    """Filas totales de un archivo/directorio parquet leyendo solo metadatos.

    Ojo: Spark escribe "archivos" mensuales como DIRECTORIOS llamados *.parquet
    con part-files dentro — por eso se filtra con is_file(). Se ignora la
    basura _temporary y los archivos de 0 bytes que deja un job interrumpido.
    """
    if not path.exists():
        return None
    files = (
        [path]
        if path.is_file()
        else [
            f for f in path.rglob("*.parquet")
            if f.is_file() and f.stat().st_size > 0 and "_temporary" not in str(f)
        ]
    )
    if not files:
        return None
    return sum(pq.ParquetFile(f).metadata.num_rows for f in files)


def dir_mb(path: Path) -> float:
    """Peso en MB de un archivo o directorio."""
    if not path.exists():
        return 0.0
    files = [path] if path.is_file() else [f for f in path.rglob("*") if f.is_file()]
    return round(sum(f.stat().st_size for f in files) / 1024**2, 2)

## Inventario de salidas gold

In [ ]:
inventario = []
for sub in ("marts", "ml"):
    base = DATA / "gold" / sub
    for d in sorted(p for p in base.iterdir() if p.is_dir()):
        inventario.append({"tipo": sub, "tabla": d.name, "filas": parquet_rows(d), "peso_mb": dir_mb(d)})
inv = pl.DataFrame(inventario)
print(f"gold total: {inv['peso_mb'].sum():,.0f} MB")
inv

## Cuadre de conteos: cada viaje contado exactamente una vez

financial/operational/tipping excluyen `fhv` **por diseño** (esa categoría
no publica tarifas ni distancias), por eso suman 292M y no 317M.

In [ ]:
MART_CATS = {
    "mart_demand_volume": CATS,
    "mart_financial_performance": ["green", "yellow", "fhvhv"],
    "mart_operational_profile": ["green", "yellow", "fhvhv"],
    "mart_tipping_behavior": ["green", "yellow", "fhvhv"],
}
cuadre = []
for mart, cats in MART_CATS.items():
    suma = (pl.scan_parquet(str(DATA / "gold" / "marts" / mart / "**" / "*.parquet"), hive_partitioning=True)
              .select(pl.col("viajes").sum()).collect().item())
    facts = sum(parquet_rows(DATA / "silver" / "star" / "facts" / f"fact_{c}_trip") or 0 for c in cats)
    cuadre.append({"mart": mart, "sum_viajes": suma, "filas_facts": facts, "cuadra": suma == facts})
pl.DataFrame(cuadre)

## Vista previa de cada mart

In [ ]:
for mart in sorted((DATA / "gold" / "marts").iterdir()):
    if not mart.is_dir():
        continue
    df = pl.read_parquet(str(mart / "**" / "*.parquet"), hive_partitioning=True, n_rows=3)
    print(f"\n=== {mart.name} ({len(df.columns)} columnas) ===")
    print(df.head(3))

## Análisis de ejemplo 1: demanda mensual por servicio (millones de viajes)

In [ ]:
demanda = (pl.scan_parquet(str(DATA / "gold" / "marts" / "mart_demand_volume" / "**" / "*.parquet"),
                           hive_partitioning=True)
    .group_by("service_id", "month").agg((pl.col("viajes").sum() / 1e6).round(2).alias("millones"))
    .collect()
    .pivot(values="millones", index="month", on="service_id")
    .sort("month"))
demanda

## Análisis de ejemplo 2: top 10 zonas por demanda anual

In [ ]:
(pl.scan_parquet(str(DATA / "gold" / "marts" / "mart_demand_volume" / "**" / "*.parquet"),
                 hive_partitioning=True)
   .group_by("pu_zone", "pu_borough").agg(pl.col("viajes").sum().alias("viajes"))
   .sort("viajes", descending=True).head(10).collect())

## Feature stores ML

- `ml_feat_isolation_fraud`: yellow+green **completos** (el modelo puntúa fraude viaje a viaje).
- `ml_feat_kmodes_trips`: yellow+green completos + **muestra 5% de fhvhv** (el modelo K-Modes entrena con máx. 100k filas por servicio; la muestra es 120× eso).
- `ml_feat_arima_trips`: serie agregada zona×hora para pronóstico.

In [ ]:
# Preview solo de los 3 FEATURE STORES (ml_feat_*). Los directorios de
# salidas de modelos (--gold-ml: kmodes_model, ml_isolation_fraud_scores,
# ml_sarimax_trips_forecast) usan otros esquemas de particion y se listan aparte.
for store in sorted((DATA / "gold" / "ml").glob("ml_feat_*")):
    if not store.is_dir():
        continue
    df = pl.read_parquet(str(store / "**" / "*.parquet"), hive_partitioning=True, n_rows=3)
    print(f"\n=== {store.name}: {parquet_rows(store):,} filas | {dir_mb(store):,.0f} MB ===")
    print(df.head(3))

print("\n--- salidas de modelos entrenados (--gold-ml) ---")
for d in sorted((DATA / "gold" / "ml").iterdir()):
    if d.is_dir() and not d.name.startswith("ml_feat_"):
        n = parquet_rows(d)
        print(f"{d.name}: {n:,} filas | {dir_mb(d):,.1f} MB" if n else f"{d.name}: {dir_mb(d):,.1f} MB")